# NY Taxi 2021 - Download & Spark
Scarica i dati yellow e green del 2021, poi li analizza con Spark.

In [1]:
import os
import urllib.request

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR = 2021
DATA_DIR = "/home/jovyan/work/data"

os.makedirs(f"{DATA_DIR}/yellow", exist_ok=True)
os.makedirs(f"{DATA_DIR}/green", exist_ok=True)

def download_if_missing(color, year, month):
    filename = f"{color}_tripdata_{year}-{month:02d}.parquet"
    dest = f"{DATA_DIR}/{color}/{filename}"
    if os.path.exists(dest):
        print(f"[skip] {filename}")
        return
    url = f"{BASE_URL}/{filename}"
    print(f"[download] {filename} ...", end=" ")
    urllib.request.urlretrieve(url, dest)
    size_mb = os.path.getsize(dest) / 1_000_000
    print(f"{size_mb:.1f} MB")

for month in range(1, 13):
    download_if_missing("yellow", YEAR, month)

for month in range(1, 13):
    download_if_missing("green", YEAR, month)

print("\nDone.")

## Spark Session

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ny-taxi-2021") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark

## Carica i dati

In [2]:
DATA_DIR = "/home/jovyan/work/data"
df_yellow = spark.read.parquet(f"{DATA_DIR}/yellow/*.parquet")
df_green  = spark.read.parquet(f"{DATA_DIR}/green/*.parquet")

print("Yellow:", df_yellow.count(), "righe")
print("Green: ", df_green.count(), "righe")

Yellow: 30904308 righe
Green:  1068755 righe


In [4]:
df_yellow.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [5]:
df_green.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: integer (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



## Esplorazione rapida

In [6]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2021-10-01 00:25:56|  2021-10-01 01:11:35|            1.0|          7.4|       1.0|                 Y|         140|          36|           1|       33.0|  3.0|    0.5|       4.

In [7]:
df_green.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2021-09-30 18:39:03|  2021-09-30 18:39:06|                 N|       5.0|          37|          37|            1.0|         0.02|       10.0|  0.0|    0.

## Unifica yellow e green con colonna `service_type`

In [3]:
from pyspark.sql import functions as F

# Rinomina colonne datetime per uniformarle
df_yellow_clean = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
    .withColumn("service_type", F.lit("yellow")) \
    .select("service_type", "pickup_datetime", "dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount")

df_green_clean = df_green \
    .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime") \
    .withColumn("service_type", F.lit("green")) \
    .select("service_type", "pickup_datetime", "dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount")

df_trips = df_yellow_clean.union(df_green_clean)
df_trips.cache()

print("Totale corse:", df_trips.count())

Totale corse: 31973063


## Aggregazioni mensili per tipo di servizio

In [4]:
df_monthly = df_trips \
    .withColumn("month", F.month("pickup_datetime")) \
    .groupBy("service_type", "month") \
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("trip_distance").alias("avg_distance_km"),
        F.avg("total_amount").alias("avg_fare_usd")
    ) \
    .orderBy("service_type", "month")

df_monthly.show(24)

+------------+-----+---------+------------------+------------------+
|service_type|month|num_trips|   avg_distance_km|      avg_fare_usd|
+------------+-----+---------+------------------+------------------+
|       green|    1|    76536|40.853433024981975|23.571431744540238|
|       green|    2|    64570|17.961487378039333|23.438734241908037|
|       green|    3|    83831|19.991962639119244|23.721150290468078|
|       green|    4|    86931| 61.92406218725194|24.079490745536987|
|       green|    5|    88183|140.31237630835926|  23.8435045303566|
|       green|    6|    86747| 201.2409743276422|23.613980541119897|
|       green|    7|    83675|194.39036593964707| 24.20218810875934|
|       green|    8|    83491| 249.7884132421469|24.520268651715813|
|       green|    9|    95711| 205.4537483674819| 25.25277000554643|
|       green|   10|   110891|167.61449738932686|23.967219972775744|
|       green|   11|   108228|207.15947277968706| 23.78228406698019|
|       green|   12|    99961| 187

In [4]:
import os
import urllib.request

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR = 2025
DATA_DIR = "/home/jovyan/work/data"

filename = f"yellow_tripdata_{YEAR}-11.parquet"
dest = f"{DATA_DIR}/yellow/{filename}"
url = f"{BASE_URL}/{filename}"
urllib.request.urlretrieve(url, dest)
size_mb = os.path.getsize(dest) / 1_000_000
print(f"{size_mb:.1f} MB")



71.1 MB


In [2]:
from pyspark.sql import SparkSession
import os

repo_filename= f"yellow_tripdata_repartitioned"
DATA_DIR = "/home/jovyan/work/data"
dest = f"{DATA_DIR}/yellow/{repo_filename}/"


spark = SparkSession.builder \
    .appName("ny-taxi-2021") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()


df = spark.read \
    .format("parquet") \
    .load("/home/jovyan/work/data/yellow/yellow_tripdata_2025-11.parquet")

df = df.repartition(4)

#df.write.parquet(dest)



In [4]:
from pyspark.sql import functions as F

df_filtered = df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15")
print(df_filtered.count())

df.createOrReplaceTempView("trips")
trip_count = spark.sql("SELECT COUNT(*) as count FROM trips WHERE DATE(tpep_pickup_datetime) = '2025-11-15'")
trip_count.show()

162604
+------+
| count|
+------+
|162604|
+------+



In [ ]:
df = df.withColumn('trip_duration', (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))/(3600))
df= df.orderBy("trip_duration", ascending=False)
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|    trip_duration|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|              1|       

In [11]:
!mkdir -p /home/jovyan/work/data/misc
!wget -P /home/jovyan/work/data/misc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 12:49:52--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.9.23.179, 65.9.23.7, 65.9.23.184, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.9.23.179|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘/home/jovyan/work/data/misc/taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-09 12:49:52 (13.9 MB/s) - ‘/home/jovyan/work/data/misc/taxi_zone_lookup.csv’ saved [12331/12331]



In [16]:
df_zones = spark.read.csv("/home/jovyan/work/data/misc/taxi_zone_lookup.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")
result = spark.sql(" SELECT zone, count(*) as num_trips FROM trips JOIN zones ON trips.PULocationID = zones.LocationID GROUP BY zone ORDER BY num_trips ASC")
result.show(10)

+--------------------+---------+
|                zone|num_trips|
+--------------------+---------+
|Governor's Island...|        1|
|Eltingville/Annad...|        1|
|       Arden Heights|        1|
|       Port Richmond|        3|
|       Rikers Island|        4|
|   Rossville/Woodrow|        4|
|         Great Kills|        4|
| Green-Wood Cemetery|        4|
|         Jamaica Bay|        5|
|         Westerleigh|       12|
+--------------------+---------+
only showing top 10 rows

